# Job C — Corruption Robustness (Colab driver)

Drives the PLAN.md §1.3 corruption-robustness benchmark on Colab. The grid is
(1 model × 1 dataset × N categories × N corruptions × N severities); each
(corruption × severity) cell shells out to `scripts/run_jobC.sh`, which
iterates over the requested categories internally.

Outputs land at `/content/work/runs/jobC_<dataset>_<cat>_<model>_<corruption>_s<sev>_<ts>/`
and are rsynced to Drive at `/content/drive/MyDrive/thesis_runs/jobC/`. Per-cell
`.done` markers live alongside the synced runs so the grid resumes correctly
across notebook sessions — delete a marker to force a re-run.

## 1 — Mount Drive
Required for both the Real-IAD zips and the Deceuninck dataset, and for
syncing the run outputs back to a persistent location.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 — Clone or pull the repo
Public GitHub mirror is used so no auth is required from Colab.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/LorenzSF/Real-time-visual-defect-detection.git'
REPO_DIR = '/content/Real-time-visual-defect-detection'

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
print('repo at:', REPO_DIR)

## 3 — Install project dependencies
Editable install pulls everything from `pyproject.toml` (anomalib, torch,
opencv, etc.). This takes ~3–5 min on a fresh Colab runtime.

In [ ]:
!pip install -q -e {REPO_DIR}

## 4 — Environment paths
Defaults match the SH script's Colab defaults; override here if your Drive
layout differs.

In [ ]:
import os

os.environ['REPO_DIR']        = REPO_DIR
os.environ['WORK_DIR']        = '/content/work'
os.environ['RIAD_ZIPS_DIR']   = '/content/drive/MyDrive/Real-IAD_dataset/realiad_1024'
os.environ['DECEU_DIR']       = '/content/drive/MyDrive/Deceuninck_dataset'
os.environ['RESULTS_DIR']     = '/content/drive/MyDrive/thesis_runs/jobC'
# OUTPUT_ROOT, CONFIG_*, MARKERS_DIR, PYTHON_CMD inherit the SH defaults.

## 5 — Run parameters
Edit the lists below to shape the grid. The cell after this one shells out
to `scripts/run_jobC.sh` once per (corruption × severity); each call iterates
over `CATEGORIES` internally (Real-IAD only).

* **MODEL**: any registered model name. PaDiM is the cheapest of the three
  feature-based candidates and is recommended as the first model to sweep.
* **DATASET**: `'realiad'` or `'deceuninck'`. Deceuninck ignores `CATEGORIES`.
* **CATEGORIES**: 5-cat subset chosen for variance across object types.
  Override to widen or narrow the Real-IAD coverage.
* **CORRUPTIONS** × **SEVERITIES**: PLAN.md §1.3 grid — 3 × 3 = 9 cells
  per category.

In [ ]:
MODEL       = 'anomalib_padim'
DATASET     = 'realiad'  # 'realiad' or 'deceuninck'
CATEGORIES  = ['audiojack', 'button_battery', 'plastic_plug', 'regulator', 'woodstick']
CORRUPTIONS = ['gaussian_blur', 'motion_blur', 'jpeg_compression']
SEVERITIES  = [1, 3, 5]

## 6 — Run the grid
Each iteration shells out to `bash scripts/run_jobC.sh`. A failure in one
cell does not abort the loop — the failure is logged and the next cell
starts. Re-running the cell skips finished cells via `.done` markers.

In [ ]:
import shlex, subprocess, time

script = f'{REPO_DIR}/scripts/run_jobC.sh'
failures = []
t_start = time.time()

for corruption in CORRUPTIONS:
    for severity in SEVERITIES:
        cmd = ['bash', script, MODEL, DATASET, corruption, str(severity)]
        if DATASET == 'realiad':
            cmd.extend(CATEGORIES)
        print('\n>', ' '.join(shlex.quote(c) for c in cmd), flush=True)
        rc = subprocess.call(cmd)
        if rc != 0:
            failures.append((corruption, severity, rc))
            print(f'[grid] cell failed (rc={rc}): {corruption} s{severity}', flush=True)

elapsed = time.time() - t_start
print(f'\n[grid] done in {elapsed/60:.1f} min')
if failures:
    print(f'[grid] {len(failures)} cell(s) failed:', failures)
else:
    print('[grid] all cells succeeded')

## 7 — Release the runtime
Frees the GPU and disconnects so Colab quota stops being charged. Skip this
cell if you plan to launch another `MODEL` sweep without restarting.

In [ ]:
import gc
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as exc:
    print(f'runtime.unassign skipped: {exc}')